
# QoS Analysis for YAFS Results

This notebook computes common QoS metrics from YAFS simulation outputs:
- End-to-End delay (mean, p50, p95) overall and per app
- Throughput (req/s) overall and per app
- Jitter (std dev of E2E delay) overall and per app
- Hop count statistics (estimated from link traces)
- Load distribution (events per destination node, and per node+module)
- Network usage (total bytes and per link)
- Compute-time utilization per node (proxy from `COMP_*` events)

**Optional (if available):**
- Meet-deadline rate & Deadline Miss Ratio (requires `data/appDeadlines.json`)
- Cloud offload rate (requires mapping of cloud node IDs)

> Update the `CONFIG` cell if your filenames or paths differ.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
from typing import Optional, Dict, Any

# === CONFIG ===
BASE = Path("results/")
EVENTS_CSV = BASE / "results_CN_10000.csv"
LINKS_CSV  = BASE / "results_CN_10000_link.csv"

# Optional: deadlines file. If present, DMR will be computed.
DEADLINES_JSON = BASE / "data" / "appDeadlines.json"

# Optional: identify cloud nodes to compute Cloud Offload Rate
# Example: CLOUD_NODE_IDS = {9999}
CLOUD_NODE_IDS: Optional[set[int]] = None

assert EVENTS_CSV.exists(), f"Events CSV not found: {EVENTS_CSV}"
assert LINKS_CSV.exists(),  f"Links CSV not found: {LINKS_CSV}"

events = pd.read_csv(EVENTS_CSV)
links  = pd.read_csv(LINKS_CSV)

events.head(3)


,id,type,app,module,message,DES.src,DES.dst,TOPO.src,TOPO.dst,module.src,service,time_in,time_out,time_emit,time_reception
0,2,COMP_M,12,12_74,M.USER.APP.12,41,356,86,100,NaN,5.4007,160.409451,165.810151,108.000000,160.409451
1,2,COMP_M,12,12_75,12_(74-75),356,363,100,100,12_74,2.9452,165.810151,168.755351,165.810151,165.810151
2,2,COMP_M,12,12_76,12_(74-76),356,370,100,100,12_74,5.8986,165.810151,171.708751,165.810151,165.810151


In [2]:

def pick_time_columns(df):
    t_in_candidates  = [c for c in df.columns if c.lower() in ("time_in","time_emit","t_in","start_time")]
    t_out_candidates = [c for c in df.columns if c.lower() in ("time_out","time_reception","t_out","end_time")]
    if not t_in_candidates or not t_out_candidates:
        raise KeyError(f"Time columns not found. Columns: {list(df.columns)}")
    return t_in_candidates[0], t_out_candidates[0]

def compute_flows(df: pd.DataFrame) -> pd.DataFrame:
    tin, tout = pick_time_columns(df)
    grp = df.groupby(["id"], as_index=False).agg(
        app=("app", "first"),
        start=(tin, "min"),
        end=(tout, "max"),
        n_events=("id","size"),
    )
    grp["e2e_delay"] = grp["end"] - grp["start"]
    grp["complete"] = np.isfinite(grp["start"]) & np.isfinite(grp["end"]) & (grp["e2e_delay"] >= 0)
    return grp

def summarize_e2e(flows: pd.DataFrame) -> dict:
    ok = flows["complete"]
    d = {
        "flows_total": int(len(flows)),
        "flows_complete": int(ok.sum()),
        "completion_rate": float(ok.mean()) if len(flows) else float("nan"),
        "mean_e2e": float(flows.loc[ok, "e2e_delay"].mean()) if ok.any() else float("nan"),
        "p50_e2e": float(flows.loc[ok, "e2e_delay"].quantile(0.50)) if ok.any() else float("nan"),
        "p95_e2e": float(flows.loc[ok, "e2e_delay"].quantile(0.95)) if ok.any() else float("nan"),
        "overall_jitter_std": float(flows.loc[ok, "e2e_delay"].std()) if ok.any() else float("nan"),
    }
    if ok.any():
        sim_time = flows.loc[ok, "end"].max() - flows.loc[ok, "start"].min()
        d["overall_throughput_req_per_s"] = float(len(flows.loc[ok]) / sim_time) if sim_time > 0 else float("nan")
    else:
        d["overall_throughput_req_per_s"] = float("nan")
    return d

def per_app_tables(flows: pd.DataFrame):
    ok = flows["complete"]
    per_app_e2e = flows.loc[ok].groupby("app").agg(
        n_flows=("id","count"),
        mean_e2e=("e2e_delay","mean"),
        p50_e2e=("e2e_delay",lambda s: s.quantile(0.50)),
        p95_e2e=("e2e_delay",lambda s: s.quantile(0.95)),
        jitter_std=("e2e_delay","std")
    ).reset_index().sort_values("app")

    per_app_throughput = flows.loc[ok].groupby("app").apply(
        lambda g: len(g) / (g["end"].max() - g["start"].min()) if (g["end"].max() - g["start"].min()) > 0 else np.nan
    ).reset_index(name="throughput_req_per_s")
    return per_app_e2e, per_app_throughput

def hop_stats(links: pd.DataFrame):
    if not {"message","ctime"}.issubset(set(links.columns)):
        return {"avg_hops": np.nan, "p50_hops": np.nan, "p95_hops": np.nan, "paths_counted": 0},                pd.DataFrame(columns=["app","n_paths","avg_hops","p50_hops","p95_hops"])
    hop_groups = links.groupby(["app","message","ctime"]).size().reset_index(name="hops_count")
    hop_overall = {
        "avg_hops": float(hop_groups["hops_count"].mean()),
        "p50_hops": float(hop_groups["hops_count"].quantile(0.50)),
        "p95_hops": float(hop_groups["hops_count"].quantile(0.95)),
        "paths_counted": int(len(hop_groups))
    }
    hop_per_app = hop_groups.groupby("app")["hops_count"].agg(["count","mean","median",lambda s: s.quantile(0.95)]).reset_index()
    hop_per_app.columns = ["app","n_paths","avg_hops","p50_hops","p95_hops"]
    return hop_overall, hop_per_app

def load_distribution(events: pd.DataFrame):
    # Events per compute destination node
    load_by_node = events.groupby(["DES.dst"]).size().reset_index(name="n_events").sort_values("n_events", ascending=False)
    # Events per node+module
    load_by_node_module = events.groupby(["DES.dst","module"]).size().reset_index(name="n_events").sort_values("n_events", ascending=False)
    return load_by_node, load_by_node_module

def network_usage(links: pd.DataFrame):
    if {"size","src","dst"}.issubset(set(links.columns)):
        total_bytes = int(links["size"].sum())
        by_link = links.groupby(["src","dst"])["size"].sum().reset_index().sort_values("size", ascending=False)
        return total_bytes, by_link
    return None, pd.DataFrame(columns=["src","dst","size"])

def compute_utilization(events: pd.DataFrame):
    # Proxy: sum of (time_out - time_in) for COMP_* events per node
    cols = set(events.columns)
    if {"time_in","time_out","DES.dst","type"}.issubset(cols):
        comp = events[events["type"].astype(str).str.contains("COMP", na=False)].copy()
        comp["comp_time"] = comp["time_out"] - comp["time_in"]
        util_by_node = comp.groupby("DES.dst")["comp_time"].sum().reset_index().sort_values("comp_time", ascending=False)
        return util_by_node
    return pd.DataFrame(columns=["DES.dst","comp_time"])

def attach_deadlines(flows: pd.DataFrame, deadlines_path: Path):
    if not deadlines_path.exists():
        flows["deadline"] = np.nan
        flows["meet_deadline"] = np.nan
        return None, None, pd.DataFrame(columns=["app","meet_deadline_rate"])
    with open(deadlines_path) as f:
        dl = json.load(f)
    def get_deadline(app):
        if isinstance(dl, list):
            return dl[app] if app < len(dl) else None
        if isinstance(dl, dict):
            return dl.get(str(app), dl.get(int(app), None))
        return None
    flows["deadline"] = flows["app"].apply(get_deadline)
    flows["meet_deadline"] = flows["complete"] & (flows["e2e_delay"] <= flows["deadline"])
    subset = flows["deadline"].notnull()
    if subset.any():
        meet_rate_overall = float(flows.loc[subset, "meet_deadline"].mean())
        dmr_overall = 1.0 - meet_rate_overall
        meet_per_app = flows.loc[subset].groupby("app")["meet_deadline"].mean().reset_index(name="meet_deadline_rate")
    else:
        meet_rate_overall = None
        dmr_overall = None
        meet_per_app = pd.DataFrame(columns=["app","meet_deadline_rate"])
    return meet_rate_overall, dmr_overall, meet_per_app

def cloud_offload_rate(events: pd.DataFrame, cloud_ids: Optional[set]) -> Optional[float]:
    if not cloud_ids:
        return None
    if "DES.dst" not in events.columns:
        return None
    # count events executed on cloud nodes (proxy)
    total = len(events)
    on_cloud = int(events["DES.dst"].isin(cloud_ids).sum())
    return on_cloud / total if total else None


In [3]:

flows = compute_flows(events)

overall = summarize_e2e(flows)
per_app_e2e, per_app_throughput = per_app_tables(flows)

hop_overall, hop_per_app = hop_stats(links)
load_by_node, load_by_node_module = load_distribution(events)
net_total_bytes, net_by_link = network_usage(links)
util_by_node = compute_utilization(events)

meet_rate_overall, dmr_overall, meet_per_app = attach_deadlines(flows, DEADLINES_JSON)
cloud_offload = cloud_offload_rate(events, CLOUD_NODE_IDS)

overall.update({
    "hop_avg": hop_overall["avg_hops"],
    "hop_p50": hop_overall["p50_hops"],
    "hop_p95": hop_overall["p95_hops"],
    "paths_counted": hop_overall["paths_counted"],
    "net_total_bytes": net_total_bytes,
    "meet_deadline_rate": meet_rate_overall,
    "deadline_miss_ratio": dmr_overall,
    "cloud_offload_rate": cloud_offload
})

overall


/var/folders/wv/xmd6tbzd5ng0cxlyp0swk29m0000gn/T/ipykernel_2639/1523327348.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_app_throughput = flows.loc[ok].groupby("app").apply(


{'flows_total': 1298,
 'flows_complete': 1298,
 'completion_rate': 1.0,
 'mean_e2e': 392.4992522808553,
 'p50_e2e': 15.900300000001153,
 'p95_e2e': 2635.000230021374,
 'overall_jitter_std': 1138.7822899245095,
 'overall_throughput_req_per_s': 0.13143030549841575,
 'hop_avg': 1.012313282196205,
 'hop_p50': 1.0,
 'hop_p95': 1.0,
 'paths_counted': 4954,
 'net_total_bytes': 14699078722,
 'meet_deadline_rate': None,
 'deadline_miss_ratio': None,
 'cloud_offload_rate': None}

In [4]:

from caas_jupyter_tools import display_dataframe_to_user

display_dataframe_to_user("Per-app E2E delay (mean/p50/p95/jitter)", per_app_e2e)
display_dataframe_to_user("Per-app throughput (req/s)", per_app_throughput)
display_dataframe_to_user("Per-app hop count (estimated from link trace)", hop_per_app)
display_dataframe_to_user("Load by node (top destinations)", load_by_node.head(50))
display_dataframe_to_user("Load by node & module (top)", load_by_node_module.head(100))
display_dataframe_to_user("Network usage by link (total bytes, top)", net_by_link.head(100))
display_dataframe_to_user("Compute-time utilization per node", util_by_node)

# Also show deadline meet rates if available
if meet_per_app is not None and not meet_per_app.empty:
    display_dataframe_to_user("Per-app meet-deadline rate", meet_per_app)


ModuleNotFoundError: No module named 'caas_jupyter_tools'

In [ ]:

import json
from pathlib import Path

OUT_DIR = Path("/mnt/data/qos_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save overall summary
with open(OUT_DIR / "overall_summary.json", "w") as f:
    json.dump(overall, f, indent=2)

# Save tables
per_app_e2e.to_csv(OUT_DIR / "per_app_e2e.csv", index=False)
per_app_throughput.to_csv(OUT_DIR / "per_app_throughput.csv", index=False)
hop_per_app.to_csv(OUT_DIR / "per_app_hops.csv", index=False)
load_by_node.to_csv(OUT_DIR / "load_by_node.csv", index=False)
load_by_node_module.to_csv(OUT_DIR / "load_by_node_module.csv", index=False)
net_by_link.to_csv(OUT_DIR / "network_usage_by_link.csv", index=False)
util_by_node.to_csv(OUT_DIR / "compute_utilization_by_node.csv", index=False)
if meet_per_app is not None and not meet_per_app.empty:
    meet_per_app.to_csv(OUT_DIR / "meet_deadline_rate_per_app.csv", index=False)

OUT_DIR, list(OUT_DIR.iterdir())
